# NB 08 — Three-material damping cluster, $K_{\\rm soft}$ envelopes, Section-4 triples

Brings together the locked Turn-4B inputs: the $\eta=\Gamma_{\rm HWHM}/\Omega$ cluster (two underdamped quantum paraelectrics STO/KTaO$_3$ + the overdamped ferroelectric BTO, empty interior), the particle-like soft-mode kernel `K_soft` evaluated with each material's $\Lambda^2$ uncertainty envelope, and the curated `section4_eta_kappa_triples.csv` anchor table that feeds the Section-4 fit. Consumes `fekappa.eta` and `fekappa.kappa_kernel`.


In [ ]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
FIG_DIR = PROJECT_ROOT / "figures"; FIG_DIR.mkdir(exist_ok=True)
PROC_DIR = PROJECT_ROOT / "data" / "processed"; PROC_DIR.mkdir(parents=True, exist_ok=True)

from fekappa.plots import set_publication_style
from fekappa import literature_io as lit
from fekappa import eta as eta_mod
from fekappa import kappa_kernel as kk
set_publication_style()

FLAGS = []
def flag(item, ok, note=""):
    print(f"  [{'OK ' if ok else 'STOP'}] {item}" + (f"  ({note})" if note else "")); FLAGS.append((item, ok, note))

print("PROJECT_ROOT:", PROJECT_ROOT)


## 1. Three-material $\eta$ cluster (two underdamped QPE + overdamped BTO)


In [ ]:
# STO + KTaO3 eta(T) from eta.py (Vogt gamma/2 HWHM); BTO eta=Gamma/Omega from HAS Table I (cubic).
Ts, es = eta_mod.eta_sto_vogt()
Tk, ek = eta_mod.eta_kto_vogt()
eta_STO, eta_KTO = float(es[-1]), float(ek[-1])

has = lit.load_harada_axe_shirane_1971_mode_parameters()
bto_to = (has[(has["mode"] == "TO") & (has["q_label"] == 0.2)]
          .drop_duplicates("T_K").sort_values("T_K"))
bto_eta = (bto_to["gamma_meV"] / bto_to["omega_meV"]).to_numpy()
eta_BTO = float(np.mean(bto_eta))
print(f"eta_STO(@{Ts[-1]:.0f}K)={eta_STO:.3f}   eta_KTO(@{Tk[-1]:.0f}K)={eta_KTO:.3f}   "
      f"eta_BTO(mean cubic)={eta_BTO:.2f}  per-T={np.round(bto_eta,2)}")

THR = 0.5
flag("STO underdamped", eta_STO < THR, f"eta={eta_STO:.3f}")
flag("KTaO3 underdamped", eta_KTO < THR, f"eta={eta_KTO:.3f}")
flag("BTO overdamped", eta_BTO > THR, f"eta={eta_BTO:.2f}")
flag("two-QPE cluster (|delta|<0.05)", abs(eta_STO - eta_KTO) < 0.05, f"|delta|={abs(eta_STO-eta_KTO):.3f}")
INTERIOR = (0.5, 2.0)
interior_empty = not any(INTERIOR[0] < e < INTERIOR[1] for e in (eta_STO, eta_KTO, eta_BTO))
flag("eta-axis interior (0.5,2.0) empty", interior_empty,
     "Section-4 fit interior EXTRAPOLATED, not anchored")

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.axvspan(*INTERIOR, color="0.9", label="empty interior (extrapolated)")
ax.axvline(THR, ls="--", color="0.5", lw=1)
ax.scatter([eta_STO, eta_KTO], [1, 1], s=90, color="tab:blue", zorder=3, label="QPE (underdamped)")
ax.scatter([eta_BTO], [1], s=90, color="tab:red", zorder=3, label="BTO (overdamped)")
for x, lab in [(eta_STO, "STO"), (eta_KTO, "KTaO$_3$"), (eta_BTO, "BTO")]:
    ax.annotate(lab, (x, 1), xytext=(0, 10), textcoords="offset points", ha="center")
ax.set_xscale("log"); ax.set_yticks([]); ax.set_ylim(0.95, 1.15)
ax.set_xlabel(r"$\eta = \Gamma_{\rm HWHM}/\Omega$")
ax.set_title("Three-material damping cluster"); ax.legend(loc="lower right", fontsize=7)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig_NB08_eta_cluster.pdf"); plt.show()


## 2. $K_{\rm soft}$ envelopes ($\Lambda^2 \pm$ frac. unc.)


In [ ]:
# Omega_0(T): STO/KTaO3 from Vogt; BTO from HAS cubic TO (near-Gamma, order-of-mag only).
sto_o = lit.load_vogt1995_sto_omega_T()
sto_o = sto_o[sto_o["phase_mode"] == "cubic_F1u"].sort_values("T_K")
kto_o = lit.load_vogt1995_kto_omega_T()
kto_o = kto_o[kto_o["method"] == "exp_HyperRaman_Vogt1995"].sort_values("T_K")

def envelope(material, T, Om, eta_val):
    """Central + lo/hi kappa_soft over the material's Lambda^2 +/- frac_unc envelope."""
    m = kk.MATERIALS[material]; f = m["lambda_sq_frac_unc"]; L2 = m["lambda_sq"]
    O2 = np.asarray(Om, float) ** 2
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # silence overdamped/skip warnings inside the sweep
        if isinstance(L2, dict):
            return {d: (kk.K_soft(O2, v, eta_val, T, m["V_cell_ang3"]),
                        kk.K_soft(O2, v * (1 - f), eta_val, T, m["V_cell_ang3"]),
                        kk.K_soft(O2, v * (1 + f), eta_val, T, m["V_cell_ang3"]))
                    for d, v in L2.items()}
        return (kk.K_soft(O2, L2, eta_val, T, m["V_cell_ang3"]),
                kk.K_soft(O2, L2 * (1 - f), eta_val, T, m["V_cell_ang3"]),
                kk.K_soft(O2, L2 * (1 + f), eta_val, T, m["V_cell_ang3"]))

Tsto = sto_o["T_K"].to_numpy(); Osto = sto_o["Omega_meV"].to_numpy()
Tkto = kto_o["T_K"].to_numpy(); Okto = kto_o["Omega_meV"].to_numpy()
Tbto = bto_to["T_K"].to_numpy(); Obto = bto_to["omega_meV"].to_numpy()
sto_ce, sto_lo, sto_hi = envelope("STO", Tsto, Osto, eta_STO)
kto_ce, kto_lo, kto_hi = envelope("KTO", Tkto, Okto, eta_KTO)
bto_env = envelope("BTO", Tbto, Obto, eta_BTO)

print(f"STO   kappa_soft @{Tsto[-1]:.0f}K = {sto_ce[-1]:.3f}  [{sto_lo[-1]:.3f}, {sto_hi[-1]:.3f}] W/m/K")
print(f"KTaO3 kappa_soft @{Tkto[-1]:.0f}K = {kto_ce[-1]:.3f}  [{kto_lo[-1]:.3f}, {kto_hi[-1]:.3f}] W/m/K")
for d, (ce, lo, hi) in bto_env.items():
    print(f"BTO[{d}] kappa_soft @{Tbto[-1]:.0f}K = {ce[-1]:.4f}  [{lo[-1]:.4f}, {hi[-1]:.4f}] W/m/K (order-of-mag)")

flag("kappa_soft << total kappa (soft-mode-only)", (sto_ce[-1] < 2.0) and (kto_ce[-1] < 2.0),
     f"STO {sto_ce[-1]:.2f}, KTaO3 {kto_ce[-1]:.2f} W/m/K vs total ~10")
flag("BTO K_soft order-of-magnitude only", all(ce[-1] > 0 for ce, _, _ in bto_env.values()),
     "overdamped: K_P invalid, deferred to Wigner K_C")

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(Tsto, sto_lo, sto_hi, color="tab:blue", alpha=0.25)
ax.plot(Tsto, sto_ce, "-o", color="tab:blue", ms=3,
        label=r"STO  ($\eta$=%.3f, $\Lambda^2$=889$\pm$30%%)" % eta_STO)
ax.fill_between(Tkto, kto_lo, kto_hi, color="tab:green", alpha=0.2)
ax.plot(Tkto, kto_ce, "-s", color="tab:green", ms=3,
        label=r"KTaO$_3$  ($\eta$=%.3f, proxy $\pm$50%%)" % eta_KTO)
ax.set_xlabel("T (K)"); ax.set_ylabel(r"$\kappa_{\rm soft}$ (W m$^{-1}$ K$^{-1}$)")
ax.set_title(r"Soft-mode particle-like $\kappa$ (K$_P$) -- underdamped QPE")
ax.legend(fontsize=7); fig.tight_layout()
fig.savefig(FIG_DIR / "fig_NB08_kappa_soft_envelopes.pdf"); plt.show()


## 3. Section-4 anchor triples $\to$ `section4_eta_kappa_triples.csv`


In [ ]:
# Section-4 anchor triples: (material, eta, kappa_soft) + provenance, evaluated at each
# material's reference temperature (highest available T per source).
rows = []
def add(material, direction, T, Om, eta_val, ce, lo, hi, regime, note):
    m = kk.MATERIALS[material]
    L2 = m["lambda_sq"][direction] if isinstance(m["lambda_sq"], dict) else m["lambda_sq"]
    rows.append(dict(material=material, direction=direction, T_K=round(float(T), 2),
        omega0_meV=round(float(Om), 3), eta=round(float(eta_val), 4),
        lambda_sq_meV2ang2=round(float(L2), 1), lambda_sq_frac_unc=m["lambda_sq_frac_unc"],
        kappa_soft_W_m_K=round(float(ce), 5), kappa_soft_lo=round(float(lo), 5),
        kappa_soft_hi=round(float(hi), 5), regime=regime, note=note))

add("STO", "isotropic", Tsto[-1], Osto[-1], eta_STO, sto_ce[-1], sto_lo[-1], sto_hi[-1],
    "underdamped", "Vogt cubic_F1u Omega(T); eta=Vogt gamma/2; Lambda2=Verdi Gamma->X-TO1 effective curvature (not small-q)")
add("KTO", "isotropic", Tkto[-1], Okto[-1], eta_KTO, kto_ce[-1], kto_lo[-1], kto_hi[-1],
    "underdamped", "Vogt Omega(T); eta=Vogt gamma/2; Lambda2=STO-proxy +/-50%")
for d, (ce, lo, hi) in bto_env.items():
    add("BTO", d, Tbto[-1], Obto[-1], eta_BTO, ce[-1], lo[-1], hi[-1],
        "overdamped(order-of-mag)", "HAS cubic TO near-Gamma; eta=Gamma/Omega; K_P invalid -> Wigner K_C")

triples = pd.DataFrame(rows)
out_csv = PROC_DIR / "section4_eta_kappa_triples.csv"
triples.to_csv(out_csv, index=False)
print(triples.to_string(index=False))
print("\nwrote", out_csv.relative_to(PROJECT_ROOT), "(data/processed/ is gitignored -> regenerates from NB 08)")

flag("Section-4 triples written", out_csv.exists(), f"{len(triples)} rows")
flag("triples span the cluster (2 QPE + BTO)", set(triples["material"]) == {"STO", "KTO", "BTO"},
     "two underdamped anchors + one overdamped, empty interior")

fig, ax = plt.subplots(figsize=(7, 4))
ax.axvspan(0.5, 2.0, color="0.92", label="empty interior")
q = triples[triples.regime == "underdamped"]
b = triples[triples.regime.str.startswith("overdamped")]
for sub, fmt, col, lab in [(q, "o", "tab:blue", "QPE (anchored)"),
                           (b, "s", "tab:red", "BTO (order-of-mag)")]:
    ax.errorbar(sub["eta"], sub["kappa_soft_W_m_K"],
                yerr=[sub["kappa_soft_W_m_K"] - sub["kappa_soft_lo"],
                      sub["kappa_soft_hi"] - sub["kappa_soft_W_m_K"]],
                fmt=fmt, color=col, capsize=3, label=lab)
for _, r in triples.iterrows():
    ax.annotate(f'{r["material"]}{"" if r["direction"] in ("isotropic",) else ":"+r["direction"]}',
                (r["eta"], r["kappa_soft_W_m_K"]), xytext=(4, 4), textcoords="offset points", fontsize=6)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"$\eta = \Gamma_{\rm HWHM}/\Omega$"); ax.set_ylabel(r"$\kappa_{\rm soft}$ (W m$^{-1}$ K$^{-1}$)")
ax.set_title(r"Section-4 plane: $\kappa_{\rm soft}$ vs $\eta$ (interior extrapolated)")
ax.legend(fontsize=7); fig.tight_layout()
fig.savefig(FIG_DIR / "fig_NB08_section4_plane.pdf"); plt.show()


## 4. Gate


In [ ]:
n_stop = sum(1 for _, ok, _ in FLAGS if not ok)
print("\nNB 08 flags:")
for item, ok, note in FLAGS:
    print(f"  [{'OK ' if ok else 'STOP'}] {item}" + (f"  ({note})" if note else ""))
print()
if n_stop:
    print(f"NB 08 GATE: CHECK ({n_stop} flag(s) tripped).")
else:
    print("NB 08 GATE: PASS. Two QPE (STO, KTaO3) cluster underdamped; BTO overdamped alone;")
    print("eta-axis interior empty -> Section-4 fit interior extrapolated. kappa_soft << total.")
    print("section4_eta_kappa_triples.csv written. done.")
